# MTG Vector Math — Full Training Pipeline
**Run All** (`Runtime → Run all`) and walk away. Takes ~5 min on T4 GPU.

At the end, `encoder.pkl` and `axis_scale.npy` are saved to `/content/mtg-vector-math/models/` and auto-downloaded to your device.

In [ ]:
# 1. Clone repo & install deps
!git clone --quiet https://github.com/yusuf-wadi/mtg-vector-math
%cd mtg-vector-math
!pip install -q -r requirements-train.txt

In [ ]:
# 2. Pull data files from mtg-oracle
!curl -sL 'https://github.com/yusuf-wadi/mtg-oracle/raw/main/data/oracle-slim.json.gz' -o data/oracle-slim.json.gz
!curl -sL 'https://raw.githubusercontent.com/yusuf-wadi/mtg-oracle/main/data/mtg-axes.json' -o data/mtg-axes.json

import gzip, json
with gzip.open('data/oracle-slim.json.gz') as f:
    cards = json.load(f)
print(f'oracle-slim loaded: {len(cards):,} cards')
with open('data/mtg-axes.json') as f:
    axes = json.load(f)
print(f'mtg-axes loaded: {len(axes)} axes')

In [ ]:
# 3. Generate 105D soft labels via bge-small teacher (~2 min on GPU)
!python scripts/01_generate_labels.py

In [ ]:
# 4. Train TF-IDF + MLP student encoder (~1 min)
!python scripts/02_train_encoder.py

In [ ]:
# 5. Validate — paste the output back to Perplexity when done
!python scripts/03_validate.py

In [ ]:
# 6. Download trained artifacts to your device
import os
from google.colab import files

for f in ['models/encoder.pkl', 'models/axis_scale.npy', 'data/labels.npy', 'data/card_index.json']:
    if os.path.exists(f):
        print(f'Downloading {f}...')
        files.download(f)
    else:
        print(f'MISSING: {f} — check step above for errors')